In [ ]:
# ── GIN Baseline: No hierarchy penalty, detailed violation analysis ──
import numpy as np
import pandas as pd
from collections import defaultdict, deque

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GINConv, global_add_pool, global_mean_pool

from rdkit import Chem

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ══════════════════════════════════════════════════════════════════════════════
# 1. Load data
# ══════════════════════════════════════════════════════════════════════════════
DATA_DIR = "tasks-2026/task1"

df_train = pd.read_parquet(f"{DATA_DIR}/chebi_dataset_train.parquet")
df_test = pd.read_parquet(f"{DATA_DIR}/chebi_dataset_test_empty.parquet")

label_cols = [c for c in df_train.columns if c.startswith("class_")]
num_classes = len(label_cols)

print(f"Train: {len(df_train)} molecules, {num_classes} classes")
print(f"Test:  {len(df_test)} molecules")

# ══════════════════════════════════════════════════════════════════════════════
# 2. Parse ChEBI hierarchy from .obo file (for violation analysis only)
# ══════════════════════════════════════════════════════════════════════════════
def parse_obo(path):
    child_to_parents = defaultdict(list)
    current_id = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("id: class_"):
                current_id = line.split("id: ")[1]
            elif line.startswith("is_a: class_"):
                parent = line.split("is_a: ")[1].strip()
                child_to_parents[current_id].append(parent)
    return child_to_parents

child_to_parents = parse_obo(f"{DATA_DIR}/chebi_classes.obo")

# Load class names for readable reports
class_defs = pd.read_csv(f"{DATA_DIR}/chebi_class_definitions.csv")
class_id_to_name = dict(zip(class_defs["chebi_id"], class_defs["name"]))

class_to_idx = {c: i for i, c in enumerate(label_cols)}

# parent_mask[i, j] = 1 if class j is a direct parent of class i
parent_mask = torch.zeros(num_classes, num_classes)
for child, parents in child_to_parents.items():
    if child in class_to_idx:
        for p in parents:
            if p in class_to_idx:
                parent_mask[class_to_idx[child], class_to_idx[p]] = 1.0

parent_mask_np = parent_mask.numpy()
print(f"Hierarchy: {int(parent_mask.sum())} direct parent-child edges")

# ══════════════════════════════════════════════════════════════════════════════
# 3. SMILES → PyG molecular graph
# ══════════════════════════════════════════════════════════════════════════════
ATOM_FEATURES = {
    'atomic_num': list(range(1, 119)),
    'degree': [0, 1, 2, 3, 4, 5],
    'formal_charge': [-2, -1, 0, 1, 2],
    'num_hs': [0, 1, 2, 3, 4],
    'hybridization': [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3,
        Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2,
    ],
}

def one_hot(val, allowed):
    vec = [0] * (len(allowed) + 1)
    if val in allowed:
        vec[allowed.index(val)] = 1
    else:
        vec[-1] = 1
    return vec

def atom_features(atom):
    return (
        one_hot(atom.GetAtomicNum(), ATOM_FEATURES['atomic_num'])
        + one_hot(atom.GetDegree(), ATOM_FEATURES['degree'])
        + one_hot(atom.GetFormalCharge(), ATOM_FEATURES['formal_charge'])
        + one_hot(atom.GetTotalNumHs(), ATOM_FEATURES['num_hs'])
        + one_hot(atom.GetHybridization(), ATOM_FEATURES['hybridization'])
        + [1 if atom.GetIsAromatic() else 0]
    )

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

def bond_features(bond):
    return (
        one_hot(bond.GetBondType(), BOND_TYPES)
        + [1 if bond.GetIsConjugated() else 0]
        + [1 if bond.IsInRing() else 0]
    )

def smiles_to_graph(smiles, labels=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    x = []
    for atom in mol.GetAtoms():
        x.append(atom_features(atom))
    x = torch.tensor(x, dtype=torch.float)
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index.extend([[i, j], [j, i]])
        edge_attr.extend([bf, bf])
    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 7), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if labels is not None:
        data.y = torch.tensor(labels, dtype=torch.float)
    return data

sample = smiles_to_graph("CCO")
NODE_FEAT_DIM = sample.x.shape[1]
print(f"Node feature dim: {NODE_FEAT_DIM}")

# ══════════════════════════════════════════════════════════════════════════════
# 4. Convert all molecules to graphs
# ══════════════════════════════════════════════════════════════════════════════
def build_graph_dataset(df, label_cols=None):
    graphs = []
    skipped = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Building graphs"):
        labels = row[label_cols].values.astype(np.float32) if label_cols else None
        g = smiles_to_graph(row["SMILES"], labels)
        if g is not None:
            g.mol_id = row["mol_id"]
            graphs.append(g)
        else:
            skipped += 1
    print(f"Built {len(graphs)} graphs, skipped {skipped}")
    return graphs

train_graphs = build_graph_dataset(df_train, label_cols)
test_graphs = build_graph_dataset(df_test, label_cols=None)

# ══════════════════════════════════════════════════════════════════════════════
# 5. Train/val split & dataloaders
# ══════════════════════════════════════════════════════════════════════════════
train_data, val_data = train_test_split(train_graphs, test_size=0.1, random_state=42)

BATCH_SIZE = 128
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_graphs)}")

# ══════════════════════════════════════════════════════════════════════════════
# 6. GIN Model (same architecture, NO hierarchy enforcement)
# ══════════════════════════════════════════════════════════════════════════════
class GINBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )
        self.conv = GINConv(mlp, train_eps=True)
        self.bn = nn.BatchNorm1d(out_dim)

    def forward(self, x, edge_index):
        x = self.conv(x, edge_index)
        x = self.bn(x)
        return F.relu(x)


class BaselineGIN(nn.Module):
    """Same GIN architecture but with NO hierarchy enforcement."""
    def __init__(self, node_feat_dim, hidden_dim, num_classes, num_layers=5, dropout=0.3):
        super().__init__()
        self.num_layers = num_layers
        self.dropout = dropout
        self.input_proj = nn.Linear(node_feat_dim, hidden_dim)
        self.gin_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.gin_layers.append(GINBlock(hidden_dim, hidden_dim))
        self.jk_proj = nn.Linear(hidden_dim * (num_layers + 1), hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.input_proj(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        layer_outputs = [x]
        for gin in self.gin_layers:
            x = gin(x, edge_index)
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_outputs.append(x)
        x = torch.cat(layer_outputs, dim=-1)
        x = self.jk_proj(x)
        x_add = global_add_pool(x, batch)
        x_mean = global_mean_pool(x, batch)
        x = torch.cat([x_add, x_mean], dim=-1)
        return self.classifier(x)


HIDDEN_DIM = 256
NUM_LAYERS = 5
DROPOUT = 0.3

model = BaselineGIN(
    node_feat_dim=NODE_FEAT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=num_classes,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# ══════════════════════════════════════════════════════════════════════════════
# 7. Plain BCE loss (NO hierarchy penalty)
# ══════════════════════════════════════════════════════════════════════════════
all_labels = np.array([g.y.numpy() for g in train_graphs])
pos_freq = all_labels.mean(axis=0)
pos_freq = np.clip(pos_freq, 0.001, 0.999)
pos_weight = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float).to(device)
pos_weight = torch.clamp(pos_weight, max=50.0)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Pos weight range: [{pos_weight.min():.2f}, {pos_weight.max():.2f}]")

# ══════════════════════════════════════════════════════════════════════════════
# 8. Violation analysis helpers
# ══════════════════════════════════════════════════════════════════════════════
def count_violations_detailed(probs, preds, parent_mask_np, label_cols, class_id_to_name):
    """
    Detailed hierarchy violation analysis on raw predictions (no enforcement).

    Returns:
      - total violations (child=1 & parent=0 in binary preds)
      - total parent-child sample pairs checked
      - prob_violations: cases where child_prob > parent_prob (continuous)
      - per_edge breakdown: for each (child, parent) edge, how many samples violate
      - per_sample: for each sample, how many edges are violated
    """
    n_samples = preds.shape[0]

    # Binary violations: child predicted 1 but parent predicted 0
    binary_violations = 0
    # Probability violations: child_prob > parent_prob
    prob_violations = 0
    total_pairs = 0

    edge_violations = []  # (child_name, parent_name, n_binary_viols, n_prob_viols, mean_diff)
    per_sample_violations = np.zeros(n_samples)

    for child_idx in range(parent_mask_np.shape[0]):
        parent_indices = np.where(parent_mask_np[child_idx] > 0)[0]
        for p_idx in parent_indices:
            total_pairs += n_samples

            # Binary: child=1, parent=0
            bin_v = ((preds[:, child_idx] == 1) & (preds[:, p_idx] == 0)).sum()
            binary_violations += bin_v

            # Probability: child_prob > parent_prob
            prob_diff = probs[:, child_idx] - probs[:, p_idx]
            prob_v_mask = prob_diff > 0
            n_prob_v = prob_v_mask.sum()
            prob_violations += n_prob_v

            # Track per-sample
            per_sample_violations += (preds[:, child_idx] > preds[:, p_idx]).astype(float)

            if bin_v > 0 or n_prob_v > 0:
                child_name = class_id_to_name.get(label_cols[child_idx], label_cols[child_idx])
                parent_name = class_id_to_name.get(label_cols[p_idx], label_cols[p_idx])
                mean_diff = prob_diff[prob_v_mask].mean() if n_prob_v > 0 else 0.0
                edge_violations.append((
                    label_cols[child_idx], child_name,
                    label_cols[p_idx], parent_name,
                    int(bin_v), int(n_prob_v), float(mean_diff)
                ))

    return {
        "binary_violations": int(binary_violations),
        "prob_violations": int(prob_violations),
        "total_pairs": total_pairs,
        "edge_violations": sorted(edge_violations, key=lambda x: -x[4]),  # sort by binary viols
        "per_sample_violations": per_sample_violations,
    }


def print_violation_report(stats, top_n=20):
    """Print a readable violation report."""
    n_samples = len(stats["per_sample_violations"])
    total_pairs = stats["total_pairs"]

    print("=" * 80)
    print("HIERARCHY VIOLATION REPORT (NO PENALTY / NO ENFORCEMENT)")
    print("=" * 80)

    print(f"\nBinary violations (child=1, parent=0): {stats['binary_violations']:,} "
          f"/ {total_pairs:,} pairs ({stats['binary_violations']/total_pairs*100:.2f}%)")
    print(f"Probability violations (P(child) > P(parent)): {stats['prob_violations']:,} "
          f"/ {total_pairs:,} pairs ({stats['prob_violations']/total_pairs*100:.2f}%)")

    # Per-sample stats
    samples_with_violations = (stats["per_sample_violations"] > 0).sum()
    print(f"\nSamples with at least 1 violation: {samples_with_violations} "
          f"/ {n_samples} ({samples_with_violations/n_samples*100:.1f}%)")
    print(f"Mean violations per sample: {stats['per_sample_violations'].mean():.2f}")
    print(f"Max violations in a single sample: {int(stats['per_sample_violations'].max())}")

    # Top violating edges
    print(f"\n{'─' * 80}")
    print(f"TOP {top_n} MOST VIOLATED PARENT-CHILD EDGES (by binary violations):")
    print(f"{'─' * 80}")
    print(f"{'Child':<35} {'Parent':<35} {'Bin.Viols':>9} {'Prob.Viols':>10} {'MeanDiff':>8}")
    print(f"{'─' * 80}")
    for edge in stats["edge_violations"][:top_n]:
        child_id, child_name, parent_id, parent_name, bin_v, prob_v, mean_d = edge
        child_label = f"{child_name[:30]} ({child_id})"
        parent_label = f"{parent_name[:30]} ({parent_id})"
        print(f"{child_label:<35} {parent_label:<35} {bin_v:>9} {prob_v:>10} {mean_d:>8.4f}")

    print(f"\nTotal edges with any binary violation: "
          f"{sum(1 for e in stats['edge_violations'] if e[4] > 0)}")
    print(f"Total edges with any prob violation:   "
          f"{sum(1 for e in stats['edge_violations'] if e[5] > 0)}")




Using device: cpu
Train: 33668 molecules, 500 classes
Test:  11223 molecules
Hierarchy: 748 direct parent-child edges
Node feature dim: 145


Building graphs:   0%|          | 1/33668 [00:00<1:01:07,  9.18it/s][15:00:15] WARNING: not removing hydrogen atom without neighbors
[15:00:15] WARNING: not removing hydrogen atom without neighbors
[15:00:15] WARNING: not removing hydrogen atom without neighbors
Building graphs:  12%|█▏        | 4179/33668 [00:02<00:15, 1874.04it/s][15:00:17] WARNING: not removing hydrogen atom without neighbors
[15:00:17] WARNING: not removing hydrogen atom without neighbors
[15:00:17] WARNING: not removing hydrogen atom without neighbors
Building graphs:  14%|█▍        | 4766/33668 [00:02<00:15, 1917.69it/s][15:00:18] WARNING: not removing hydrogen atom without neighbors
[15:00:18] WARNING: not removing hydrogen atom without neighbors
[15:00:18] WARNING: not removing hydrogen atom without neighbors
Building graphs:  19%|█▊        | 6298/33668 [00:03<00:14, 1832.30it/s][15:00:19] WARNING: not removing hydrogen atom without neighbors
[15:00:19] WARNING: not removing hydrogen atom without neighbors
Buil

Built 33668 graphs, skipped 0


Building graphs:   8%|▊         | 936/11223 [00:00<00:04, 2366.64it/s][15:00:34] WARNING: not removing hydrogen atom without neighbors
[15:00:34] WARNING: not removing hydrogen atom without neighbors
[15:00:34] WARNING: not removing hydrogen atom without neighbors
[15:00:34] WARNING: not removing hydrogen atom without neighbors
[15:00:34] WARNING: not removing hydrogen atom without neighbors
Building graphs:  10%|█         | 1173/11223 [00:00<00:04, 2277.77it/s][15:00:34] WARNING: not removing hydrogen atom without neighbors
[15:00:34] WARNING: not removing hydrogen atom without neighbors
[15:00:34] WARNING: not removing hydrogen atom without neighbors
[15:00:34] Unusual charge on atom 6 number of radical electrons set to zero
Building graphs:  12%|█▏        | 1402/11223 [00:00<00:04, 2243.94it/s][15:00:34] WARNING: not removing hydrogen atom without neighbors
[15:00:34] WARNING: not removing hydrogen atom without neighbors
[15:00:34] WARNING: not removing hydrogen atom without neighbo

Built 11223 graphs, skipped 0
Train: 30301, Val: 3367, Test: 11223
Model parameters: 1,323,385
Pos weight range: [0.00, 50.00]


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# 9. Evaluation (raw predictions, no hierarchy enforcement)
# ══════════════════════════════════════════════════════════════════════════════
def evaluate(model, loader, threshold=0.5):
    """Evaluate with RAW predictions (no hierarchy enforcement)."""
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).float()
            all_probs.append(probs.cpu().numpy())
            labels = batch.y.view(preds.shape).cpu().numpy()
            all_labels.append(labels)
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    all_preds = (all_probs >= threshold).astype(float)

    f1_micro = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    # Quick binary violation count
    violations = 0
    total_pairs = 0
    for child_idx in range(parent_mask_np.shape[0]):
        parent_indices = np.where(parent_mask_np[child_idx] > 0)[0]
        for p_idx in parent_indices:
            violations += ((all_preds[:, child_idx] == 1) & (all_preds[:, p_idx] == 0)).sum()
            total_pairs += len(all_preds)

    return f1_micro, f1_macro, int(violations), total_pairs, all_probs, all_preds, all_labels


In [5]:
LR = 1e-3
EPOCHS = 50
PATIENCE = 8

optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

best_f1 = 0
patience_counter = 0
history = {"train_loss": [], "val_f1_micro": [], "val_f1_macro": [], "violations": [], "viol_rate": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss = criterion(logits, batch.y.view(logits.shape))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    avg_loss = total_loss / len(train_data)

    f1_micro, f1_macro, violations, total_pairs, _, _, _ = evaluate(model, val_loader)
    scheduler.step(f1_micro)

    viol_rate = violations / total_pairs * 100 if total_pairs > 0 else 0
    history["train_loss"].append(avg_loss)
    history["val_f1_micro"].append(f1_micro)
    history["val_f1_macro"].append(f1_macro)
    history["violations"].append(violations)
    history["viol_rate"].append(viol_rate)

    print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | "
          f"F1-micro: {f1_micro:.4f} | F1-macro: {f1_macro:.4f} | "
          f"Violations: {violations:,} ({viol_rate:.2f}%)")

    if f1_micro > best_f1:
        best_f1 = f1_micro
        patience_counter = 0
        torch.save(model.state_dict(), "best_model_baseline.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest validation F1-micro: {best_f1:.4f}")

Epoch 01 | Loss: 0.3962 | F1-micro: 0.5218 | F1-macro: 0.3099 | Violations: 25,661 (1.02%)
Epoch 02 | Loss: 0.3355 | F1-micro: 0.5526 | F1-macro: 0.3346 | Violations: 21,183 (0.84%)
Epoch 03 | Loss: 0.3119 | F1-micro: 0.5812 | F1-macro: 0.3606 | Violations: 21,061 (0.84%)
Epoch 04 | Loss: 0.2897 | F1-micro: 0.5832 | F1-macro: 0.3730 | Violations: 18,824 (0.75%)
Epoch 05 | Loss: 0.2754 | F1-micro: 0.6022 | F1-macro: 0.3731 | Violations: 18,373 (0.73%)
Epoch 06 | Loss: 0.2650 | F1-micro: 0.6083 | F1-macro: 0.3859 | Violations: 17,506 (0.70%)
Epoch 07 | Loss: 0.2545 | F1-micro: 0.6196 | F1-macro: 0.3903 | Violations: 17,031 (0.68%)
Epoch 08 | Loss: 0.2490 | F1-micro: 0.6290 | F1-macro: 0.4024 | Violations: 17,123 (0.68%)
Epoch 09 | Loss: 0.2426 | F1-micro: 0.6392 | F1-macro: 0.4077 | Violations: 15,237 (0.60%)
Epoch 10 | Loss: 0.2360 | F1-micro: 0.6392 | F1-macro: 0.4133 | Violations: 14,990 (0.60%)
Epoch 11 | Loss: 0.2316 | F1-micro: 0.6494 | F1-macro: 0.4197 | Violations: 14,826 (0.59%)

KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 11. Training curves
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_loss"])
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(history["val_f1_micro"], label="F1-micro")
axes[1].plot(history["val_f1_macro"], label="F1-macro")
axes[1].set_title("Validation F1")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(history["viol_rate"], color="red")
axes[2].set_title("Violation Rate % (val)")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("%")

plt.tight_layout()
plt.savefig("training_curves_baseline.png", dpi=150)
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 12. Detailed violation analysis on validation set
# ══════════════════════════════════════════════════════════════════════════════
print("\n\nLoading best model for detailed violation analysis...")
model.load_state_dict(torch.load("best_model_baseline.pt", weights_only=True))

f1_micro, f1_macro, violations, total_pairs, val_probs, val_preds, val_labels = evaluate(model, val_loader)
print(f"Best model - F1-micro: {f1_micro:.4f}, F1-macro: {f1_macro:.4f}")

stats = count_violations_detailed(val_probs, val_preds, parent_mask_np, label_cols, class_id_to_name)
print_violation_report(stats, top_n=30)

# ── Violation distribution histogram ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-sample violation distribution
sv = stats["per_sample_violations"]
axes[0].hist(sv[sv > 0], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title(f"Violations per sample (only samples with >0)\n"
                   f"{(sv > 0).sum()}/{len(sv)} samples affected")
axes[0].set_xlabel("Number of violated edges")
axes[0].set_ylabel("Number of samples")

# Per-edge violation counts
edge_bin_viols = [e[4] for e in stats["edge_violations"] if e[4] > 0]
if edge_bin_viols:
    axes[1].hist(edge_bin_viols, bins=50, edgecolor="black", alpha=0.7, color="salmon")
    axes[1].set_title(f"Binary violations per edge (only edges with >0)\n"
                       f"{len(edge_bin_viols)} edges affected")
    axes[1].set_xlabel("Number of violating samples")
    axes[1].set_ylabel("Number of edges")

plt.tight_layout()
plt.savefig("violation_analysis_baseline.png", dpi=150)
plt.show()

# ── Probability difference analysis ──
print("\n\nPROBABILITY ANALYSIS: child_prob vs parent_prob")
print("=" * 60)
all_diffs = []
for child_idx in range(parent_mask_np.shape[0]):
    parent_indices = np.where(parent_mask_np[child_idx] > 0)[0]
    for p_idx in parent_indices:
        diffs = val_probs[:, child_idx] - val_probs[:, p_idx]
        all_diffs.extend(diffs.tolist())

all_diffs = np.array(all_diffs)
print(f"Total (child, parent) probability pairs: {len(all_diffs):,}")
print(f"Cases where child_prob > parent_prob: {(all_diffs > 0).sum():,} ({(all_diffs > 0).mean()*100:.2f}%)")
print(f"Cases where child_prob > parent_prob + 0.1: {(all_diffs > 0.1).sum():,} ({(all_diffs > 0.1).mean()*100:.2f}%)")
print(f"Cases where child_prob > parent_prob + 0.3: {(all_diffs > 0.3).sum():,} ({(all_diffs > 0.3).mean()*100:.2f}%)")
print(f"Mean(child_prob - parent_prob): {all_diffs.mean():.4f}")
print(f"Mean where child > parent: {all_diffs[all_diffs > 0].mean():.4f}" if (all_diffs > 0).any() else "")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(all_diffs, bins=100, edgecolor="black", alpha=0.7)
ax.axvline(x=0, color="red", linestyle="--", label="child = parent")
ax.set_title("Distribution of (child_prob - parent_prob) across all edges & samples")
ax.set_xlabel("child_prob - parent_prob")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.savefig("prob_diff_distribution_baseline.png", dpi=150)
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 13. Generate test predictions & submission (raw, no enforcement)
# ══════════════════════════════════════════════════════════════════════════════
model.eval()
all_preds = []
all_mol_ids = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        logits = model(batch)
        preds = (torch.sigmoid(logits) >= 0.5).int()
        all_preds.append(preds.cpu().numpy())
        all_mol_ids.extend(batch.mol_id)

all_preds = np.vstack(all_preds)

submission = pd.DataFrame(all_preds, columns=label_cols)
submission.insert(0, "mol_id", all_mol_ids)
submission = df_test[["mol_id", "SMILES"]].merge(submission, on="mol_id", how="left")
submission[label_cols] = submission[label_cols].fillna(0).astype(int)

# Count violations in submission
pred_vals = submission[label_cols].values
violations = 0
for child_idx in range(parent_mask_np.shape[0]):
    parent_indices = np.where(parent_mask_np[child_idx] > 0)[0]
    for p_idx in parent_indices:
        violations += ((pred_vals[:, child_idx] == 1) & (pred_vals[:, p_idx] == 0)).sum()

print(f"\nSubmission shape: {submission.shape}")
print(f"Hierarchy violations in test submission: {violations:,}")

submission.to_parquet("chebi_submission_baseline.parquet", index=False)
print("Saved to chebi_submission_baseline.parquet")
